<a href="https://colab.research.google.com/github/tartiwiaulia/pcos-detection/blob/dev/pcos_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1
# Tujuan: load dataset PCOS dari Google Drive ke Colab
# ============================================================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_excel(
    '/content/drive/MyDrive/pcos-project/PCOS_data_without_infertility.xlsx',
    sheet_name=1
)

print("Jumlah data:", df.shape)
print("\nKolom yang tersedia:")
print(df.columns.tolist())
print("\nLima data pertama:")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Jumlah data: (541, 45)

Kolom yang tersedia:
['Sl. No', 'Patient File No.', 'PCOS (Y/N)', ' Age (yrs)', 'Weight (Kg)', 'Height(Cm) ', 'BMI', 'Blood Group', 'Pulse rate(bpm) ', 'RR (breaths/min)', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)', 'Marraige Status (Yrs)', 'Pregnant(Y/N)', 'No. of aborptions', '  I   beta-HCG(mIU/mL)', 'II    beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)', 'Unnamed: 44']

Lima data pertama:
   Sl. No

In [ ]:
# ============================================================
# CELL 2
# Tujuan: bersihkan nama kolom dari spasi aneh di awal/akhir
# ============================================================
df.columns = df.columns.str.strip()

print("Kolom setelah dibersihkan:")
print(df.columns.tolist())

Kolom setelah dibersihkan:
['Sl. No', 'Patient File No.', 'PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI', 'Blood Group', 'Pulse rate(bpm)', 'RR (breaths/min)', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)', 'Marraige Status (Yrs)', 'Pregnant(Y/N)', 'No. of aborptions', 'I   beta-HCG(mIU/mL)', 'II    beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)', 'Unnamed: 44']


In [ ]:
# ============================================================
# CELL 3
# Tujuan: pilih 8 kolom yang relevan untuk prediksi PCOS
# hanya kolom yang bisa dijawab user tanpa perlu tes lab
# ============================================================
kolom_pakai = [
    'BMI',
    'Cycle(R/I)',
    'Weight gain(Y/N)',
    'hair growth(Y/N)',
    'Pimples(Y/N)',
    'Hair loss(Y/N)',
    'Skin darkening (Y/N)',
    'PCOS (Y/N)'
]

df_clean = df[kolom_pakai].copy()
df_clean = df_clean.fillna(df_clean.mean(numeric_only=True))

print("Jumlah data:", df_clean.shape)
print("\nNilai kosong per kolom:")
print(df_clean.isnull().sum())
print("\nContoh data:")
print(df_clean.head())

Jumlah data: (541, 8)

Nilai kosong per kolom:
BMI                     0
Cycle(R/I)              0
Weight gain(Y/N)        0
hair growth(Y/N)        0
Pimples(Y/N)            0
Hair loss(Y/N)          0
Skin darkening (Y/N)    0
PCOS (Y/N)              0
dtype: int64

Contoh data:
         BMI  Cycle(R/I)  Weight gain(Y/N)  hair growth(Y/N)  Pimples(Y/N)  \
0  19.300000           2                 0                 0             0   
1  24.921163           2                 0                 0             0   
2  25.270891           2                 0                 0             1   
3  29.674945           2                 0                 0             0   
4  20.060954           2                 0                 0             0   

   Hair loss(Y/N)  Skin darkening (Y/N)  PCOS (Y/N)  
0               0                     0           0  
1               0                     0           0  
2               1                     0           1  
3               0                

In [ ]:
# ============================================================
# CELL 4
# Tujuan: cek statistik dan distribusi data
# pastikan nilai masuk akal sebelum lanjut training
# ============================================================
print(df_clean.describe())

print("\nDistribusi PCOS (Y/N):")
print(df_clean['PCOS (Y/N)'].value_counts())

print("\nNilai unik Cycle(R/I):")
print(sorted(df_clean['Cycle(R/I)'].unique()))

              BMI  Cycle(R/I)  Weight gain(Y/N)  hair growth(Y/N)  \
count  541.000000  541.000000        541.000000        541.000000   
mean    24.311285    2.560074          0.377079          0.273567   
std      4.056399    0.901950          0.485104          0.446202   
min     12.417882    2.000000          0.000000          0.000000   
25%     21.641274    2.000000          0.000000          0.000000   
50%     24.238227    2.000000          0.000000          0.000000   
75%     26.634958    4.000000          1.000000          1.000000   
max     38.900000    5.000000          1.000000          1.000000   

       Pimples(Y/N)  Hair loss(Y/N)  Skin darkening (Y/N)  PCOS (Y/N)  
count    541.000000      541.000000            541.000000  541.000000  
mean       0.489834        0.452865              0.306839    0.327172  
std        0.500359        0.498234              0.461609    0.469615  
min        0.000000        0.000000              0.000000    0.000000  
25%        0.00000

In [ ]:
# ============================================================
# CELL 5
# Tujuan: perbaiki kolom Cycle(R/I) jadi binary 0 dan 1
# 2 = teratur → 0, 4 = tidak teratur → 1, 5 = edge case → 0
# ============================================================
kolom_final = [
    'BMI',
    'Cycle(R/I)',
    'Weight gain(Y/N)',
    'hair growth(Y/N)',
    'Pimples(Y/N)',
    'Hair loss(Y/N)',
    'Skin darkening (Y/N)',
    'PCOS (Y/N)'
]

df_final = df[kolom_final].copy()

df_final['Cycle(R/I)'] = df_final['Cycle(R/I)'].apply(
    lambda x: 1 if x == 4 else 0
)

df_final = df_final.fillna(df_final.mean(numeric_only=True))

print("Jumlah data:", df_final.shape)
print("\nNilai kosong (harus semua 0):")
print(df_final.isnull().sum())
print("\nContoh data:")
print(df_final.head())

Jumlah data: (541, 8)

Nilai kosong (harus semua 0):
BMI                     0
Cycle(R/I)              0
Weight gain(Y/N)        0
hair growth(Y/N)        0
Pimples(Y/N)            0
Hair loss(Y/N)          0
Skin darkening (Y/N)    0
PCOS (Y/N)              0
dtype: int64

Contoh data:
         BMI  Cycle(R/I)  Weight gain(Y/N)  hair growth(Y/N)  Pimples(Y/N)  \
0  19.300000           0                 0                 0             0   
1  24.921163           0                 0                 0             0   
2  25.270891           0                 0                 0             1   
3  29.674945           0                 0                 0             0   
4  20.060954           0                 0                 0             0   

   Hair loss(Y/N)  Skin darkening (Y/N)  PCOS (Y/N)  
0               0                     0           0  
1               0                     0           0  
2               1                     0           1  
3               0          

In [ ]:
# ============================================================
# CELL 6
# Tujuan: cek berapa data dengan nilai Cycle = 5
# hasilnya cuma 1 data, diabaikan, lanjut training
# ============================================================
print("Jumlah data dengan Cycle(R/I) asli = 5:")
print(len(df[df['Cycle(R/I)'] == 5]))

print("\nDistribusi PCOS untuk Cycle=5:")
print(df[df['Cycle(R/I)'] == 5]['PCOS (Y/N)'].value_counts())

Jumlah data dengan Cycle(R/I) asli = 5:
1

Distribusi PCOS untuk Cycle=5:
PCOS (Y/N)
1    1
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 7
# Tujuan: training model JST (Neural Network) dari data PCOS
# split 80% training 20% testing, normalisasi, lalu evaluasi
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

X = df_final.drop('PCOS (Y/N)', axis=1)
y = df_final['PCOS (Y/N)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Akurasi model JST:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nLaporan lengkap:")
print(classification_report(y_test, y_pred,
      target_names=['PCOS Negatif', 'PCOS Positif']))

Akurasi model JST: 76.15 %

Laporan lengkap:
              precision    recall  f1-score   support

PCOS Negatif       0.80      0.88      0.84        77
PCOS Positif       0.62      0.47      0.54        32

    accuracy                           0.76       109
   macro avg       0.71      0.68      0.69       109
weighted avg       0.75      0.76      0.75       109



In [ ]:
# ============================================================
# CELL 8
# Tujuan: simpan model dan scaler ke file .pkl
# tanpa ini hasil training hilang setiap Colab restart
# ============================================================
import pickle

with open('pcos_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('pcos_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model berhasil disimpan!")
print("File: pcos_model.pkl dan pcos_scaler.pkl")

Model berhasil disimpan!
File: pcos_model.pkl dan pcos_scaler.pkl


In [ ]:
# ============================================================
# CELL 9
# Tujuan: pindahkan file .pkl ke Google Drive supaya permanen
# file ini yang nanti dipakai FastAPI untuk prediksi
# ============================================================
import shutil

shutil.copy('pcos_model.pkl',
    '/content/drive/MyDrive/pcos-project/pcos_model.pkl')

shutil.copy('pcos_scaler.pkl',
    '/content/drive/MyDrive/pcos-project/pcos_scaler.pkl')

print("Model tersimpan di Google Drive!")
print("Cek folder pcos-project di Drive kamu")

Model tersimpan di Google Drive!
Cek folder pcos-project di Drive kamu
